# Binary Classification: Linear Limits and Non-Linear Solutions

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/Bushra-Khan49/OCI_AI_Foundations_Projects/main?filepath=01_Synthetic_Nonlinear_Classification/01_mlp_make_circles.ipynb)
*Note: If you are viewing this via Binder, please run all the cells below to set up the environment before experimenting with the interactive decision boundary at the bottom.* 

In this notebook, I explore a classic problem in machine learning: when linear models fail. I use a synthetic dataset called `make_circles`, which generates data points in two concentric circles. By design, you cannot draw a straight line to separate the inner circle from the outer one.

To demonstrate this, I first fit a standard linear model (Logistic Regression) to serve as a baseline. Then, I train a small neural network (Multi-Layer Perceptron) to see how its non-linear activation functions allow it to warp the decision boundary and perfectly encapsulate the data. 
Importantly, I split the data into a training set and a validation cohort, ensuring that we evaluate these models on data they have never seen before. We don't just want a model that memorizes; we want a model that generalizes.

First, I'll import the standard libraries required for data generation, modeling, and visualization.

In [ ]:
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_circles
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 0  # fixed seed used everywhere below for reproducibility


Here, I generate the synthetic data and split it into a training set and an unseen validation cohort. I hold out 30% of the data. Everything reported below is measured on this validation set to prevent overfitting.

In [ ]:
X, y = make_circles(n_samples=300, noise=0.05, factor=0.5, random_state=RANDOM_STATE)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=RANDOM_STATE, stratify=y
)

print(f"Train samples: {len(X_train)}, Test samples: {len(X_test)}")


Next, I fit the Logistic Regression baseline. Because it attempts to find a flat, linear plane to separate the circles, we expect its accuracy to hover around 50% (equivalent to random guessing for a balanced binary problem).

In [ ]:
baseline = LogisticRegression()
baseline.fit(X_train, y_train)
baseline_acc = accuracy_score(y_test, baseline.predict(X_test))

print(f"Logistic Regression (linear baseline) test accuracy: {baseline_acc:.3f}")
print(classification_report(y_test, baseline.predict(X_test)))


Now, I introduce a neural network (MLP). By using a non-linear activation function (ReLU), the network can combine multiple flat boundaries to form a curved shape.

In [ ]:
mlp = MLPClassifier(hidden_layer_sizes=(10,), activation='relu', max_iter=3000, random_state=1)
mlp.fit(X_train, y_train)
mlp_acc = accuracy_score(y_test, mlp.predict(X_test))

print(f"MLP (hidden_layer_size=10) test accuracy: {mlp_acc:.3f}")
print(classification_report(y_test, mlp.predict(X_test)))
print("Confusion matrix:\n", confusion_matrix(y_test, mlp.predict(X_test)))


To see exactly how the network's capacity grows, I sweep through different hidden layer sizes. A network with 1 or 2 neurons might still struggle to enclose the circle smoothly, but as we increase the neurons, the validation accuracy should climb to 100%.

In [ ]:
hidden_sizes = [1, 2, 3, 5, 10, 20, 50]
sweep_results = []

for h in hidden_sizes:
    clf = MLPClassifier(hidden_layer_sizes=(h,), activation='relu', max_iter=3000, random_state=1)
    clf.fit(X_train, y_train)
    acc = accuracy_score(y_test, clf.predict(X_test))
    sweep_results.append({"hidden_layer_size": h, "test_accuracy": acc})
    print(f"hidden_layer_size={h:>3}  ->  test accuracy={acc:.3f}")


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sizes = [r["hidden_layer_size"] for r in sweep_results]
accs = [r["test_accuracy"] for r in sweep_results]
ax.plot(sizes, accs, marker='o', label='MLP')
ax.axhline(baseline_acc, color='red', linestyle='--', label=f'Logistic Regression baseline ({baseline_acc:.2f})')
ax.set_xlabel('Hidden layer size (neurons)')
ax.set_ylabel('Test accuracy')
ax.set_title('Test accuracy vs. hidden layer size (make_circles)')
ax.set_ylim(0, 1.05)
ax.legend()
fig.tight_layout()
fig.savefig(RESULTS_DIR / 'accuracy_vs_hidden_size.png', dpi=150)
plt.show()


I visualize the decision boundary of the best performing neural network. By mapping a grid of points and plotting the model's predictions, we can visually see the exact non-linear shape it learned to separate the two classes.

In [ ]:
best = max(sweep_results, key=lambda r: r["test_accuracy"])
best_h = best["hidden_layer_size"]

clf = MLPClassifier(hidden_layer_sizes=(best_h,), activation='relu', max_iter=3000, random_state=1)
clf.fit(X_train, y_train)

x_vals = np.linspace(X[:, 0].min() - 0.1, X[:, 0].max() + 0.1, 200)
y_vals = np.linspace(X[:, 1].min() - 0.1, X[:, 1].max() + 0.1, 200)
X_plane, Y_plane = np.meshgrid(x_vals, y_vals)
grid_points = np.column_stack((X_plane.ravel(), Y_plane.ravel()))
Z = clf.predict(grid_points).reshape(X_plane.shape)

fig, ax = plt.subplots(figsize=(6, 5))
ax.contourf(X_plane, Y_plane, Z, alpha=0.3)
ax.scatter(X_train[:, 0], X_train[:, 1], c=y_train, edgecolors='k', label='train')
ax.scatter(X_test[:, 0], X_test[:, 1], c=y_test, edgecolors='red', marker='s', label='test')
ax.set_title(f'Decision boundary, hidden_layer_size={best_h} (test acc={best["test_accuracy"]:.2f})')
ax.legend()
fig.tight_layout()
fig.savefig(RESULTS_DIR / 'decision_boundary.png', dpi=150)
plt.show()


### Interactive Exploration
Below is an interactive widget that lets you adjust the number of hidden neurons on the fly. As you move the slider, the neural network is retrained, and you can see how the decision boundary evolves in real-time.

In [ ]:
import ipywidgets as widgets
from IPython.display import display
from ipywidgets import interactive

def update_plot(hidden_layer_size):
    clf = MLPClassifier(hidden_layer_sizes=(hidden_layer_size,), activation='relu', max_iter=3000, random_state=1)
    clf.fit(X_train, y_train)
    acc = accuracy_score(y_test, clf.predict(X_test))

    Z = clf.predict(grid_points).reshape(X_plane.shape)
    plt.clf()
    plt.contourf(X_plane, Y_plane, Z, alpha=0.3)
    plt.scatter(X[:, 0], X[:, 1], c=y, edgecolors='k')
    plt.title(f'Hidden layer size: {hidden_layer_size}  |  test accuracy: {acc:.2f}')
    plt.show()

interactive_plot = interactive(update_plot, hidden_layer_size=widgets.IntSlider(min=1, max=50, step=1, value=10))
display(interactive_plot)


Finally, I save the trained model, the generated plots, and the performance metrics to a JSON file. Saving these artifacts is an important habit for reproducibility—allowing us or others to reference the exact results in the future without needing to re-run the entire computation.

In [ ]:
metrics = {
    "dataset": "sklearn.datasets.make_circles(n_samples=300, noise=0.05, factor=0.5, random_state=0)",
    "train_test_split": "70/30, stratified, random_state=0",
    "baseline_logistic_regression_test_accuracy": baseline_acc,
    "mlp_fixed_hidden10_test_accuracy": mlp_acc,
    "hidden_layer_sweep": sweep_results,
    "best_hidden_layer_size": best_h,
    "best_test_accuracy": best["test_accuracy"],
}

with open(RESULTS_DIR / "metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print(json.dumps(metrics, indent=2))
